# Auditoría y Limpieza del Dataset Climático (SMN)

Este notebook limpia la segunda pieza del proyecto: el registro meteorológico diario del Servicio Meteorológico Nacional, desde 1991 hasta la actualidad. Es la contraparte climática del dataset de cultivos que ya dejamos limpio en el notebook anterior — sin esto en condiciones, no hay forma de cruzar clima con rendimiento más adelante. Acá verificamos la consistencia de las 92 estaciones meteorológicas del país, resolvemos los placeholders de "sin dato" propios del SMN y confirmamos que las variables físicas (temperatura, humedad, precipitación, viento) estén dentro de rangos posibles.

## 1. Librerías, Carga y Consolidación

### 1.1 Librerías

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

### 1.2 Carga y Consolidación de Datos Meteorológicos

Se cargan las dos fuentes de datos diarios del SMN —el archivo `.xlsx` (1991-2020) y el archivo `.lst` (desde 2021, texto delimitado por tabulaciones)— y se homologa el nombre de la columna de nubosidad (`NUB_TOT` → `NUB_TOTAL`) antes de concatenarlas en un único DataFrame.

In [ ]:
RUTA_PROYECTO = Path().resolve()
if RUTA_PROYECTO.name == "notebook":  # los notebooks se ejecutan desde la carpeta notebook/
    RUTA_PROYECTO = RUTA_PROYECTO.parent
RUTA_RAW = RUTA_PROYECTO / "data" / "raw"
RUTA_INTERIM = RUTA_PROYECTO / "data" / "interim"

RUTA_INTERIM.mkdir(parents=True, exist_ok=True)

In [ ]:
df_clima_1991_2020 = pd.read_excel(
    RUTA_RAW / "datos_meteorologicos_1991_2020.xlsx",
    sheet_name="Datos meteorológicos"
)

df_clima_2021_actual = pd.read_csv(
    RUTA_RAW / "datos_meteorologicos_desde_2021.lst",
    sep="\t"
)

In [ ]:
df_clima_2021_actual = df_clima_2021_actual.rename(columns={"NUB_TOT": "NUB_TOTAL"})

df_clima_1991_2020.columns = df_clima_1991_2020.columns.str.strip()
df_clima_2021_actual.columns = df_clima_2021_actual.columns.str.strip()

# Verificación
set(df_clima_1991_2020.columns) ^ set(df_clima_2021_actual.columns)

In [ ]:
df_clima = pd.concat([df_clima_1991_2020, df_clima_2021_actual], ignore_index=True)

assert df_clima.shape[0] == len(df_clima_1991_2020) + len(df_clima_2021_actual), "Las filas no coinciden con la suma esperada"
df_clima.shape

In [ ]:
ruta_salida_interim = RUTA_INTERIM / "clima_consolidado.csv"
df_clima.to_csv(ruta_salida_interim, index=False, encoding="utf-8")

Con las dos fuentes (1991-2020 y 2021 en adelante) ya consolidadas en un solo DataFrame, toca auditarlas: ver qué tan completas están, si hay registros corruptos y qué convenciones propias del SMN tenemos que entender antes de tocar nada.

## 2. Auditoría Exploratoria de Datos Climáticos

### 2.1 Dimensiones y Tipos de Datos

In [ ]:
df_clima.shape

In [ ]:
df_clima.info()

### 2.2 Valores Nulos

In [ ]:
df_clima.isnull().sum()

In [ ]:
(df_clima.isnull().sum() / len(df_clima) * 100).round(2)

### 2.3 Duplicados

In [ ]:
# Duplicados de fila completa
df_clima.duplicated().sum()

In [ ]:
df_clima[df_clima.duplicated(keep=False)]

In [ ]:
# Duplicados sobre la clave lógica: misma estación, mismo día
df_clima.duplicated(subset=['NRO_OMM', 'FECHA']).sum()

### 2.4 Estadísticas Descriptivas

In [ ]:
df_clima.describe(include='object')

### 2.5 Valores en Cero y Placeholders Especiales

In [ ]:
# Búsqueda del placeholder 'S/D' detectado en la muestra inicial (columna HELIOF)
(df_clima == 'S/D').sum()

In [ ]:
df_clima.select_dtypes(include='number').eq(0).sum()

In [ ]:
cols_numericas_esperadas = [
    'TMAX', 'TMIN', 'TMEDIA', 'PRECIP', 'HELIOF',
    'VTO_MDIR', 'VTO_MVEL', 'VEL_MED', 'PRE_EST', 'HUM_REL', 'NUB_TOTAL'
]

for col in cols_numericas_esperadas:
    convertido = pd.to_numeric(df_clima[col], errors='coerce')
    no_numericos = df_clima[col].notna() & convertido.isna()
    print(f"{col}: {no_numericos.sum()} valores no numéricos")

In [ ]:
for col in cols_numericas_esperadas:
    convertido = pd.to_numeric(df_clima[col], errors='coerce')
    no_numericos = df_clima[col].notna() & convertido.isna()
    if no_numericos.sum() > 0:
        print(col, df_clima.loc[no_numericos, col].unique())

In [ ]:
df_clima[df_clima['TMAX'] == 'TMAX']

In [ ]:
mask_encabezado = df_clima['TMAX'] == 'TMAX'
mask_encabezado.sum()

In [ ]:
# ¿Las filas con 'S/D' en HELIOF tienen datos válidos en las demás columnas?
df_clima[df_clima['HELIOF'].str.strip() == 'S/D'][['NRO_OMM', 'NOMBRE', 'FECHA', 'TMAX', 'TMIN']].head()

### 2.6 Outliers (IQR / Boxplot)

**Pendiente.** No es posible calcular cuartiles sobre columnas de tipo `object` que mezclan texto (placeholders `'S/D'`, filas corruptas) con valores numéricos. Este análisis se retoma en la sección 4, una vez corregidos los tipos de datos en la Limpieza.

### 2.7 Rangos y Coherencia Física Inválidos

**Pendiente**, por el mismo motivo que 2.6.

### 2.8 Búsqueda de Mojibake

In [ ]:
df_clima[df_clima['NOMBRE'].str.contains('Ã', na=False)]

## Resumen de hallazgos pendientes para la sección de Limpieza

La auditoría nos deja un par de cosas puntuales para resolver:

- Eliminar las filas corruptas donde el encabezado de columna quedó cargado como si fuera un registro más.
- Reemplazar los placeholders `'S/D'` (y strings vacíos) por nulos reales en las columnas numéricas.
- Aplicar a `PRECIP` su convención propia: celda vacía significa "sin lluvia" (0 mm), no dato faltante.
- Convertir las columnas numéricas y `FECHA` a su tipo correcto, para poder calcular cuartiles y validar rangos físicos (pendiente desde `2.6` y `2.7`).

## 3. Limpieza y Corrección de Datos Climáticos

### 3.1 Eliminación de Filas Corruptas

En `2.5` encontramos filas donde el archivo trae, literalmente, el encabezado de columna repetido como si fuera un registro más (`TMAX` con el valor de texto `'TMAX'`). Las volvemos a identificar acá y las eliminamos — son basura de carga, no datos climáticos reales.

In [ ]:
mask_corrupta = df_clima['TMAX'] == 'TMAX'
mask_corrupta.sum()

In [ ]:
df_clima = df_clima[~mask_corrupta].reset_index(drop=True)

df_clima.shape

In [ ]:
# Verificación: no debería quedar ninguna fila corrupta
(df_clima['TMAX'] == 'TMAX').sum()

### 3.2 Estandarización de Placeholders de "Sin Dato"

In [ ]:
def estandarizar_placeholders(df, columnas, placeholders=('S/D', ''), valor_nulo=np.nan):
    """
    Limpia espacios en blanco sobrantes y reemplaza placeholders de "sin dato"
    por un valor nulo en las columnas indicadas de un DataFrame.

    Parámetros:
    - df: DataFrame a modificar
    - columnas: lista de nombres de columna a procesar
    - placeholders: valores de texto a tratar como "sin dato" (por defecto: 'S/D' y string vacío)
    - valor_nulo: qué usar como marcador de nulo (por defecto np.nan, adecuado para columnas float)

    Retorna:
    - El mismo DataFrame, con las columnas ya limpias
    """
    for col in columnas:
        df[col] = df[col].str.strip()
    df[list(columnas)] = df[list(columnas)].replace({p: valor_nulo for p in placeholders})
    return df

In [ ]:
cols_placeholder_generico = [
    'TMAX', 'TMIN', 'TMEDIA', 'HELIOF',
    'VTO_MDIR', 'VTO_MVEL', 'VEL_MED', 'PRE_EST', 'HUM_REL', 'NUB_TOTAL'
]

df_clima = estandarizar_placeholders(df_clima, cols_placeholder_generico)

In [ ]:
# Verificación: ya no debería quedar ningún valor no numérico (fuera de los NaN esperados)
for col in cols_placeholder_generico:
    convertido = pd.to_numeric(df_clima[col], errors='coerce')
    no_numericos = df_clima[col].notna() & convertido.isna()
    print(f"{col}: {no_numericos.sum()} valores no numéricos")

In [ ]:
# PRECIP tiene una convención propia según el diccionario oficial del SMN:
# Celda vacía desde el origen (NaN nativo al momento de la carga) significa "sin precipitación" = 0 mm
df_clima['PRECIP'] = df_clima['PRECIP'].fillna(0)

# No son equivalentes, así que se tratan por separado del resto de las columnas.
df_clima['PRECIP'] = df_clima['PRECIP'].apply(lambda x: x.strip() if isinstance(x, str) else x)
df_clima['PRECIP'] = df_clima['PRECIP'].replace({'S/D': np.nan, '': 0})

In [ ]:
# Verificación
convertido = pd.to_numeric(df_clima['PRECIP'], errors='coerce')
no_numericos = df_clima['PRECIP'].notna() & convertido.isna()
no_numericos.sum()

**Hallazgo y corrección — `PRECIP`:** la variable presentaba un altísimo porcentaje de valores nulos (~83%) que, en un primer análisis, parecía una limitación estructural del registro histórico de precipitación. Sin embargo, el diccionario de datos oficial del SMN aclara que, para esta variable específicamente, **una celda vacía significa "sin precipitación" (0 mm)**, mientras que **`'S/D'` es el único código que representa una falta de dato real**. Aplicando esta distinción, la proporción de datos genuinamente faltantes se redujo a apenas 0,23% (2.694 días sobre 1.171.911), consistente con la cobertura casi completa de las demás variables climáticas.

### 3.3 Corrección de Tipos de Datos

In [ ]:
cols_numericas = [
    'TMAX', 'TMIN', 'TMEDIA', 'PRECIP', 'HELIOF',
    'VTO_MDIR', 'VTO_MVEL', 'VEL_MED', 'PRE_EST', 'HUM_REL', 'NUB_TOTAL'
]

df_clima[cols_numericas] = df_clima[cols_numericas].astype(float)
df_clima['NRO_OMM'] = pd.to_numeric(df_clima['NRO_OMM'], errors='coerce').astype('Int64')

# Unificación de una variante real (no solo espacios en blanco)
df_clima['NOMBRE'] = df_clima['NOMBRE'].replace({'PRESIDENCIA ROQUE SAENZ PEÑA': 'PRESIDENCIA ROQUE SAENZ PEÑA AERO'})
# Limpieza de espacios en blanco sobrantes en el nombre de estación
df_clima['NOMBRE'] = df_clima['NOMBRE'].str.strip()
df_clima['NOMBRE'] = df_clima['NOMBRE'].astype('category')

In [ ]:
cruce = df_clima.groupby('NRO_OMM')['NOMBRE'].nunique()
(cruce > 1).sum()

In [ ]:
df_clima['NRO_OMM'].isna().sum()

### 3.4 Corrección de la Columna FECHA

In [ ]:
fechas_texto = df_clima.loc[df_clima['FECHA'].apply(lambda x: isinstance(x, str)), 'FECHA']

fechas_texto[fechas_texto.str.startswith(tuple(str(n) for n in range(13, 32)))].unique()[:10]

In [ ]:
df_clima['FECHA'] = pd.to_datetime(df_clima['FECHA'], dayfirst=True, errors='coerce')

In [ ]:
# Verificación: no debería haber quedado ninguna fecha sin poder convertirse
df_clima['FECHA'].isna().sum()

In [ ]:
df_clima['FECHA'].dtype

### 3.5 Diagnóstico de Valores Nulos Remanentes

In [ ]:
# Verificación
print("PRECIP = 0 (sin precipitación):", (df_clima['PRECIP'] == 0).sum())
print("PRECIP nulo ('S/D', sin dato real):", df_clima['PRECIP'].isna().sum())

In [ ]:
df_clima['anio'] = df_clima['FECHA'].dt.year

cols_dudosas = ['VTO_MDIR', 'VTO_MVEL', 'VEL_MED', 'HUM_REL', 'NUB_TOTAL']

df_clima.groupby('anio')[cols_dudosas].apply(lambda x: x.notna().mean() * 100).round(1)

In [ ]:
tabla_precip_anio = df_clima.groupby('anio')['PRECIP'].apply(lambda x: x.notna().mean() * 100).round(1)
print(tabla_precip_anio.to_string())

anio
1991     99.3
1992     99.3
1993     99.4
1994     99.5
1995     99.4
1996     99.8
1997     99.8
1998     99.9
1999     99.8
2000     99.8
2001     99.7
2002     99.8
2003     99.8
2004     99.8
2005     99.8
2006     99.9
2007     99.7
2008     99.7
2009     99.7
2010     99.8
2011     99.9
2012    100.0
2013    100.0
2014    100.0
2015    100.0
2016    100.0
2017    100.0
2018     99.9
2019     99.9
2020     99.9
2021     99.9
2022     99.6
2023     99.7
2024     99.8
2025     99.8
2026     99.7


**Nota:** la tabla muestra, para cada año, el porcentaje de registros con `PRECIP` informado (no `'S/D'`). Se imprime completa porque el entorno trunca las salidas largas.

In [ ]:
tabla_precip_estacion = df_clima.groupby('NOMBRE', observed=True)['PRECIP'].apply(lambda x: x.notna().mean() * 100).round(1)

tabla_precip_estacion.describe()

In [ ]:
ceros = (df_clima['PRECIP'] == 0).sum()
reportados = df_clima['PRECIP'].notna().sum()

print(f"Valores en cero: {ceros} ({ceros/reportados*100:.1f}% de los días reportados)")
print(f"Valores positivos (con lluvia): {reportados - ceros} ({(reportados-ceros)/reportados*100:.1f}% de los días reportados)")

### 3.6 Verificación de Duplicados

Los 2 duplicados detectados en la Auditoría (`2.3`) corresponden a las mismas 3 filas corruptas identificadas en `2.5` (encabezados repetidos, ver `3.1`) — al ser idénticas entre sí en las 14 columnas, `pandas` marcaba 2 de las 3 como duplicadas de la primera. Al eliminarlas en `3.1`, el hallazgo de `2.3` quedó resuelto como efecto secundario. Se verifica a continuación que no queda ningún duplicado real por la clave lógica (estación + fecha).

In [ ]:
df_clima[df_clima.duplicated(subset=['NRO_OMM', 'FECHA'], keep=False)]

### 3.7 Outliers y Rangos Inválidos

Retomamos los análisis pendientes de `2.6` y `2.7`, ahora que todas las columnas numéricas tienen su tipo correcto.

In [ ]:
cols_iqr = ['TMAX', 'TMIN', 'TMEDIA', 'PRECIP', 'HELIOF', 'VTO_MVEL', 'VEL_MED', 'PRE_EST', 'HUM_REL']

for col in cols_iqr:
    q1 = df_clima[col].quantile(0.25)
    q3 = df_clima[col].quantile(0.75)
    iqr = q3 - q1
    limite_inferior = q1 - 1.5 * iqr
    limite_superior = q3 + 1.5 * iqr
    outliers = df_clima[(df_clima[col] < limite_inferior) | (df_clima[col] > limite_superior)]
    print(f"{col}: {len(outliers)} outliers ({len(outliers) / df_clima[col].notna().sum() * 100:.1f}%)")

**Nota sobre `PRECIP`:** al tener una gran proporción de ceros (días sin lluvia) y una cola larga hacia valores altos (lluvias intensas), el método IQR va a marcar como "outlier" una cantidad importante de días con lluvia real. Esto es esperable en variables de precipitación y no es, por sí solo, un error de carga — solo lo trataríamos como problema si aparecieran valores físicamente imposibles (negativos, por ejemplo).

In [ ]:
# TMIN no puede ser mayor a TMAX
inconsistente_temp = df_clima[df_clima['TMIN'] > df_clima['TMAX']]
len(inconsistente_temp)

In [ ]:
# Humedad relativa fuera de 0-100%
hum_invalida = df_clima[(df_clima['HUM_REL'] < 0) | (df_clima['HUM_REL'] > 100)]
len(hum_invalida)

In [ ]:
# Precipitación negativa (físicamente imposible)
precip_negativa = df_clima[df_clima['PRECIP'] < 0]
len(precip_negativa)

In [ ]:
# Nubosidad fuera de la escala de octavos (0 a 8)
nub_invalida = df_clima[(df_clima['NUB_TOTAL'] < 0) | (df_clima['NUB_TOTAL'] > 8)]
len(nub_invalida)

In [ ]:
# Dirección de viento fuera de los códigos válidos (0=calma, 99=variable, o 1-36 en decenas de grado)
vto_invalido = df_clima[~df_clima['VTO_MDIR'].isin([0, 99]) & ~df_clima['VTO_MDIR'].between(1, 36)]
len(vto_invalido)

In [ ]:
vto_invalido = df_clima[
    df_clima['VTO_MDIR'].notna() &
    ~df_clima['VTO_MDIR'].isin([0, 99]) &
    ~df_clima['VTO_MDIR'].between(1, 36)
]
len(vto_invalido)

**Nota:** el primer intento de este chequeo (`~df_clima['VTO_MDIR'].isin([0, 99]) & ~df_clima['VTO_MDIR'].between(1, 36)`, sin excluir explícitamente los valores nulos) devolvió `997.618` filas "inválidas" — un ~85% del dataset, prácticamente idéntico a la proporción de `NaN` que ya conocíamos para esta variable. La causa: `.isin()` y `.between()` devuelven `False` ante un valor `NaN`, y al negarlos con `~` ese `False` se convierte en `True`, marcando por error los valores faltantes como "fuera de rango". Se corrige agregando `df_clima['VTO_MDIR'].notna()` a la condición, lo que redujo el resultado a `0` — confirmando que no hay direcciones de viento realmente inválidas en el dataset.

**Resultado de `3.7`:** ningún chequeo de coherencia física (`TMIN > TMAX`, `HUM_REL`, `PRECIP` negativo, `NUB_TOTAL`, `VTO_MDIR`) detectó valores inválidos. En outliers (IQR), la mayoría de las variables muestra una proporción baja y esperable (0.0%-1.9%), salvo `PRECIP` (21.4%, ya documentado por su naturaleza zero-inflada) y, en menor medida, `PRE_EST` (6.2%) y `VEL_MED` (4.6%), que probablemente reflejan la diversidad real de altitud y ubicación entre las 92 estaciones del país, no errores de carga. No se aplican correcciones adicionales en esta sección.

## 4. Cierre de la Fase de Limpieza

Con esto, el registro diario del SMN entre 1991 y 2026 — más de 1,1 millones de observaciones de 92 estaciones — queda auditado, corregido y con un tipo de dato correcto en cada columna. En el camino eliminamos filas corruptas (encabezados repetidos como si fueran datos), aclaramos la convención real de `PRECIP` (celda vacía = sin lluvia, `'S/D'` = dato realmente faltante) y confirmamos que ningún valor viola una regla física básica (temperaturas, humedad, nubosidad o dirección de viento fuera de rango).

Vale la pena tener presente una limitación real para lo que sigue: `VTO_MVEL`, `VEL_MED`, `HUM_REL` y `NUB_TOTAL` solo están disponibles de forma consistente desde 2021 — antes de eso, el SMN no las relevaba de la misma manera. No es un error de este dataset, es una limitación de la fuente histórica, y hay que tenerla en cuenta en cualquier análisis que mire décadas anteriores a 2021.

El dataset limpio se guarda en `data/processed`, listo para cruzarse con los datos de cultivos en el próximo notebook.

In [ ]:
df_clima.info()

In [ ]:
df_clima.head()

In [ ]:
RUTA_PROCESSED = RUTA_PROYECTO / "data" / "processed"
RUTA_PROCESSED.mkdir(parents=True, exist_ok=True)

ruta_salida = RUTA_PROCESSED / "clima_procesado.csv"
df_clima.to_csv(ruta_salida, index=False, encoding='utf-8')

print(f"Guardado en: {ruta_salida}")
print(f"Dimensiones finales: {df_clima.shape}")